# Third-review GPU run on Kaggle (C1, C2, X1, C4)

Runs `correction/run_review3_kaggle.py`: the float32 check of the two float16 models (C1), fairer prompts
for the Bengali checkpoints (C2), the Grad-CAM attribution (X1) and the 2x2 decoding ablation (C4).
See `context/plan-draft-03.md` and `context/decisions.md` D36-D38.

**Before running:**
1. Session options: **Accelerator = GPU T4 x2** (two GPUs cost the same quota as one), Internet = on.
2. Add data: the private dataset that holds `ocr_eval40.jsonl` (the one used for the earlier runs).
   To resume a stopped run, also add that run's output (it contains a `review3/` folder).
3. Add-ons, Secrets: `HF_TOKEN` (Gemma and Llama are gated).
4. Set `HOURS` in the first code cell to your remaining GPU quota minus about 45 minutes.
5. Save Version, then "Save & Run All (Commit)". It runs in the background; the browser can be closed.

When it finishes, download `review3.zip` from the version's Output tab.


In [ ]:
import glob, os, shutil, subprocess, sys, time

# Remaining Kaggle GPU quota minus a margin for setup and saving the output.
# Quota on 2026-10-15: 10 h 34 min, so 9.25 h of jobs.
HOURS = 9.25

REPO_DIR = "/tmp/bengali-postocr-sllm"
OUT_DIR = "/kaggle/working/review3"
os.makedirs(OUT_DIR, exist_ok=True)
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "-q",
                    "https://github.com/Ariffurrhmn/bengali-postocr-sllm.git", REPO_DIR], check=True)
# transformers pinned to the version of the main runs (10-12 October 2026).
subprocess.run("pip install -q transformers==5.19.0 accelerate sentencepiece jiwer protobuf "
               "git+https://github.com/csebuetnlp/normalizer", shell=True, check=True)
commit = subprocess.run(["git", "-C", REPO_DIR, "log", "--oneline", "-1"], capture_output=True, text=True).stdout
import torch, transformers
env = (f"repo commit: {commit}python: {sys.version.split()[0]}\ntorch: {torch.__version__}\n"
       f"cuda: {torch.version.cuda}\ncudnn: {torch.backends.cudnn.version()}\n"
       f"transformers: {transformers.__version__}\ngpus: {[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]}\n")
open(f"{OUT_DIR}/environment.txt", "w").write(env)
print(env)
assert torch.cuda.device_count() == 2, "Set Accelerator to GPU T4 x2"


In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
login(token=UserSecretsClient().get_secret("HF_TOKEN"))


In [ ]:
ocr_files = glob.glob("/kaggle/input/**/ocr_eval40.jsonl", recursive=True)
assert ocr_files, "Attach the dataset that contains ocr_eval40.jsonl"
OCR = ocr_files[0]
print("OCR file:", OCR, "pages:", sum(1 for _ in open(OCR, encoding="utf-8")))


In [ ]:
# All jobs, in priority order; progress lines appear here, details in review3/*.log.
subprocess.run([sys.executable, "run_review3_kaggle.py", "--ocr-path", OCR, "--out-dir", OUT_DIR,
                "--hours", str(HOURS)], cwd=f"{REPO_DIR}/correction")


In [ ]:
import collections, json
for f in sorted(glob.glob(f"{OUT_DIR}/*.jsonl")):
    rows = [json.loads(l) for l in open(f, encoding="utf-8") if l.strip()]
    errs = sum(1 for r in rows if r.get("error"))
    keys = collections.Counter((r.get("model_key") or r.get("model"), r.get("engine")) for r in rows)
    print(os.path.basename(f), len(rows), "rows", f"errors {errs}" if errs else "", dict(keys))
shutil.make_archive("/kaggle/working/review3", "zip", "/kaggle/working", "review3")
print("wrote /kaggle/working/review3.zip")
